# HPT prototype on an A100 — llama.cpp for the app, vLLM for the 600-case eval

Two generators, both **in this process**, no server and no port:

| who | backend | why |
|---|---|---|
| the app (UI, live queries) | `llama_cpp.Llama` | the answer prompt carries the whole step trace and 40+ MRF rows — 8,193 tokens on a mammogram question. llama.cpp rolls its window; vLLM hard-rejects anything over `max_model_len`. |
| the 600-case eval | `vllm.LLM` | fixed, short prompts; paged KV and CUDA graphs turn 2,400 generations into minutes. |

Run the app cells first. The eval cell loads vLLM **beside** llama.cpp, so it asks
for less GPU (`gpu_memory_utilization=0.70`); if it complains about memory, stop
the UI first.

## 1. Code and dependencies

In [ ]:
%cd /content
import os
if not os.path.isdir("/content/HPT_Prototype"):
    !git clone https://github.com/adi274903/HPT_Prototype /content/HPT_Prototype
%cd /content/HPT_Prototype

!pip install -q --upgrade pip
!pip install -q duckdb sentence-transformers qdrant-client pandas

# llama.cpp with CUDA offload, prebuilt wheel for this image's CUDA
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

!git log --oneline -1          # expect 40454bd or later

If the wheel index has nothing for the image's CUDA, build it once (≈10 min):

```python
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --no-binary :all:
```

## 2. Drive, Qdrant snapshot, MRF

In [ ]:
import colab_run
colab_run.mount_drive()
qdrant_path = colab_run.stage_db()
print(colab_run.default_paths())

## 3. The app's generator — llama.cpp, GGUF, GPU layers offloaded

In [ ]:
import os
GGUF = "/content/models/medgemma-1.5-4b-it-Q8_0.gguf"
os.makedirs("/content/models", exist_ok=True)
if not os.path.exists(GGUF):
    !wget -q -O $GGUF https://huggingface.co/unsloth/medgemma-1.5-4b-it-GGUF/resolve/main/medgemma-1.5-4b-it-Q8_0.gguf

from pt_healthcare.backends import LlamaCppPipe

app_model = LlamaCppPipe(
    GGUF,
    n_ctx=16384,        # the answer prompt is ~8.2k and grows with the price table
    n_gpu_layers=-1,    # every layer on the A100
    n_threads=8,
)

pipeline = colab_run.setup(mount=False, stage=False, qdrant_path=qdrant_path, pipe=app_model)
print(pipeline.run("What might a diagnostic mammogram cost at UPMC Presbyterian?")["answer"])

## 4. Serve the UI

In [ ]:
ui = colab_run.serve_ui(pipeline=pipeline, page="patient", tunnel="cloudflare")
print(ui.pages)        # {'patient': 'https://<id>.trycloudflare.com/', 'dashboard': '.../pipeline'}
# ui.stop()  when done — the tunnel is public and unauthenticated

## 5. The eval's generator — vLLM, 600 fixed prompts

In [ ]:
import os
if not os.path.exists(".cache/mrf_eval.duckdb"):     # once, ~8 min over the 2.3 GB CSV
    !python scripts/prepare_eval_mrf.py --mrf "/content/drive/MyDrive/Price_Transparency/datasets/everyUPMCmrf_clean (1).csv" --out .cache/mrf_eval.duckdb

In [ ]:
from pt_healthcare.eval_runner import evaluate, format_report

# Its own instance, next to llama.cpp: 0.70 of 40 GB still leaves ~19 GB of KV.
# qdrant_path is where the catalog comes from — same store the app queries, so no
# ~/Downloads/*_embed_ready.csv is needed. Stop the UI first if memory is tight.
report = evaluate(
    backend="vllm",
    model="google/medgemma-1.5-4b-it",
    qdrant_path=qdrant_path,
    max_model_len=32768,             # eval prompts are short; the cap never bites
    gpu_memory_utilization=0.70,
)
print(format_report(report))
report["codes"]["dice_mrf"], report["rows"]["dice_raw"], report["ner"]["hospital"]["f1"]

A ten-case smoke before the full run:

```python
report = evaluate(backend="vllm", per_subtype=1, gpu_memory_utilization=0.70)
```

The eval appends every finished case to `eval_run.jsonl` and skips what is already
there, so a dropped runtime costs the cases in flight. `fresh=True` starts over.

Run order matters: llama.cpp owns the card until you free it (`del app_model,
pipeline; gc.collect(); torch.cuda.empty_cache()`), and vLLM needs its own slice.
If both are resident, expect the UI to stall while eval generations run — they
share one GPU.